## KV Cache：为什么已经有 $W_K$，还需要保存 $K$？

**先区分三个东西：**

- **$W_K$（参数矩阵）**：模型训练时不断更新，**训练结束后会保存在模型权重中**。
- **$K=XW_K$（本次计算的 Key）**：由当前输入 $X$ 和 $W_K$ 计算得到。训练时可能为反向传播暂时保留，但**不会作为训练好的模型参数长期保存**。
- **KV Cache（推理缓存）**：在**同一次自回归生成过程**中，把已经算出的 $K$、$V$ 临时保存，下一轮直接复用；它不是新的可训练参数。

### ① 训练：学习并保存 $W_K$，而不是保存「苹果」的 $K$

训练样本是「苹果 很 甜」。监督关系是：**「苹果」预测「很」；「苹果 很」预测「甜」**。这只是按前缀理解训练目标；实际 GPT 可以通过因果掩码在一次前向传播中并行计算这些预测。训练时模型计算 $K=XW_K$，再根据损失更新 $W_K$ 等参数。**训练结束保存的是 $W_K$、$W_V$、$W_Q$ 等模型参数，不是这个训练样本算出的 $K$、$V$。**

### ② 推理：用户输入「苹果」，假设模型生成「很」

为了看清楚重复计算，假设训练后的参数和两词的输入特征如下（数值仅作演示）：

$$
W_K=\begin{bmatrix}2&1\\1&3\end{bmatrix},\quad
x_{\text{苹果}}=\begin{bmatrix}1&0\end{bmatrix},\quad
x_{\text{很}}=\begin{bmatrix}0&1\end{bmatrix}.
$$

**不使用 KV Cache：每一轮重新处理整个已知上文。**

$$
\begin{aligned}
\text{第1轮（输入「苹果」）：}\quad
K^{(1)}&=\underbrace{\begin{bmatrix}1&0\end{bmatrix}}_{X^{(1)}}W_K
=\begin{bmatrix}2&1\end{bmatrix}
\quad\Longrightarrow\quad\text{预测「很」}\\[8pt]
\text{第2轮（输入「苹果 很」）：}\quad
K^{(2)}&=\underbrace{\begin{bmatrix}1&0\\0&1\end{bmatrix}}_{X^{(2)}}W_K
=\begin{bmatrix}2&1\\1&3\end{bmatrix}
\quad\Longrightarrow\quad\text{预测「甜」}.
\end{aligned}
$$

注意两次计算结果中的 $[2,1]$：**这是「苹果」的 K，被计算了两遍**。第二轮的 $W_K$ 没变，但如果不保留上一轮算出的 $K$，就得重新调用 $W_K$ 计算历史输入的 K；$V$ 也是如此。

**使用 KV Cache：第一轮保存结果，第二轮只算新增 Token。**

$$
\begin{aligned}
\text{第1轮缓存：}\quad K_{\text{cache}}&=\begin{bmatrix}2&1\end{bmatrix}\\[6pt]
\text{第2轮新计算：}\quad k_{\text{很}}&=\begin{bmatrix}0&1\end{bmatrix}W_K=\begin{bmatrix}1&3\end{bmatrix}\\[6pt]
\text{第2轮使用的 K：}\quad K&=\operatorname{concat}(K_{\text{cache}},k_{\text{很}})
=\begin{bmatrix}2&1\\1&3\end{bmatrix}.
\end{aligned}
$$

这时**只需计算「很」的 Query**，但仍需使用「苹果、很」的 Key 和 Value，所以可有 $Q:(1,d_k)$、$K:(2,d_k)$、$V:(2,d_v)$。历史 Q 不需要缓存；历史 K、V 要留给新 Query 访问。

**结论：$W_K$ 是训练后保存、供所有输入使用的固定参数；$K$ 是每次根据实际输入算出来的结果。KV Cache 就是在推理时额外保存历史 $K$、$V$，避免下一轮重复计算。** 实际 Transformer 会在每个注意力层分别维护缓存，生成结束后通常可以释放。